# NBER Environment and Energy Economics Knowledge Graph

This notebook turns the structured NBER paper dataset into a typed knowledge graph. It creates paper, author, topic, method, and data-source nodes, then connects them with meaningful relationships.

The core graph construction uses only Python's standard library and the tested `src/build_nber_knowledge_graph.py` module. The final 3D visualization is optional.

## 1. Locate the workspace

The notebook works when Jupyter starts in the repository root or any of its subdirectories.

In [ ]:
from pathlib import Path
import sys

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        builder = candidate / 'src' / 'build_nber_knowledge_graph.py'
        sample = candidate / 'data' / 'sample_papers.csv'
        if builder.exists() and sample.exists():
            return candidate
    raise FileNotFoundError(
        'Run Jupyter from the repository root or a subdirectory of it.'
    )

ROOT = find_repository_root(Path.cwd().resolve())
FULL_INPUT = ROOT / 'outputs' / 'nber_environment_energy_structured.csv'
SAMPLE_INPUT = ROOT / 'data' / 'sample_papers.csv'
INPUT_CSV = FULL_INPUT if FULL_INPUT.exists() else SAMPLE_INPUT
BUILDER_DIR = ROOT / 'src'
sys.path.insert(0, str(BUILDER_DIR))

print('Repository:', ROOT)
print('Input:', INPUT_CSV)


## 2. Import the knowledge-graph builder

The builder contains the topic, method, and data-source dictionaries plus the graph-construction and export functions.

In [ ]:
import build_nber_knowledge_graph as kg

print('Topics:', len(kg.TOPIC_KEYWORDS))
print('Methods:', len(kg.METHOD_KEYWORDS))
print('Data-source categories:', len(kg.DATA_KEYWORDS))

## 3. Load the structured paper data

Each row represents one paper and contains authors, dates, source links, research-question notes, findings, contributions, data signals, and methods.

In [ ]:
rows = kg.read_rows(INPUT_CSV)
print(f'Loaded {len(rows):,} papers')

sample = rows[0]
for key in ['paper_number', 'title', 'authors', 'research_question', 'main_findings']:
    print(f"\n{key}: {sample.get(key, '')}")

## 4. Understand the graph schema

Node types:

- `paper`
- `author`
- `topic`
- `method`
- `data_source`

Edge types:

- `AUTHORED_BY`
- `HAS_TOPIC`
- `USES_METHOD`
- `USES_DATA_SIGNAL`
- `SHARES_AUTHOR_WITH`

## 5. Build the knowledge graph

In [ ]:
nodes, edges = kg.build_graph(rows)
print(f'Nodes: {len(nodes):,}')
print(f'Edges: {len(edges):,}')

## 6. Inspect node and edge counts

In [ ]:
from collections import Counter

node_type_counts = Counter(node['type'] for node in nodes)
edge_type_counts = Counter(edge['type'] for edge in edges)

print('Node types')
for node_type, count in node_type_counts.most_common():
    print(f'  {node_type:15s} {count:>8,}')

print('\nEdge types')
for edge_type, count in edge_type_counts.most_common():
    print(f'  {edge_type:20s} {count:>8,}')

## 7. Inspect example relationships

In [ ]:
node_lookup = {node['id']: node for node in nodes}

for edge in edges[:12]:
    source = node_lookup[edge['source']]['label']
    target = node_lookup[edge['target']]['label']
    print(f"{source} --[{edge['type']}]--> {target}")

## 8. Explore one topic

Change `focus_topic` to another topic key from `kg.TOPIC_KEYWORDS`.

In [ ]:
focus_topic = 'climate_policy'
topic_node_id = kg.node_id('topic', focus_topic)

paper_ids = [
    edge['source']
    for edge in edges
    if edge['type'] == 'HAS_TOPIC' and edge['target'] == topic_node_id
]

print(f"Papers connected to '{focus_topic}': {len(paper_ids):,}")
for paper_id in paper_ids[:15]:
    print('-', node_lookup[paper_id]['label'])

## 9. Export the graph

The CSV files work well in Gephi, Cytoscape, Neo4j import workflows, and custom visualizations. GraphML can be opened directly in many graph tools.

In [ ]:
kg.write_csv(
    kg.NODES_CSV,
    nodes,
    ['id', 'label', 'type', 'paper_number', 'display_date',
     'publication_date', 'doi', 'url', 'pdf_url', 'abstract_word_count'],
)

kg.write_csv(
    kg.EDGES_CSV,
    edges,
    ['id', 'source', 'target', 'type', 'weight', 'matched_terms', 'shared_author'],
)

kg.write_graphml(kg.GRAPHML, nodes, edges)

print('Wrote:', kg.NODES_CSV)
print('Wrote:', kg.EDGES_CSV)
print('Wrote:', kg.GRAPHML)

## 10. Optional interactive 3D preview

This section requires NetworkX and Plotly. In a normal Jupyter environment, install them once with:

```python
%pip install networkx plotly
```

The preview intentionally uses a small topic-centered subgraph. Drawing all 29,000+ edges interactively would be slow and visually overwhelming.

In [ ]:
# Uncomment in a Jupyter environment if these packages are missing.
# %pip install networkx plotly

In [ ]:
try:
    import networkx as nx
    import plotly.graph_objects as go
except ImportError:
    print('Install optional packages with: %pip install networkx plotly')
else:
    max_papers = 80
    selected_papers = set(paper_ids[:max_papers])
    selected_nodes = selected_papers | {topic_node_id}

    # Add authors, methods, data sources, and topics attached to selected papers.
    for edge in edges:
        if edge['source'] in selected_papers and edge['type'] in {
            'AUTHORED_BY', 'HAS_TOPIC', 'USES_METHOD', 'USES_DATA_SIGNAL'
        }:
            selected_nodes.add(edge['target'])

    selected_edges = [
        edge for edge in edges
        if edge['source'] in selected_nodes and edge['target'] in selected_nodes
        and edge['type'] != 'SHARES_AUTHOR_WITH'
    ]

    graph = nx.Graph()
    for node_id_value in selected_nodes:
        graph.add_node(node_id_value, **node_lookup[node_id_value])
    for edge in selected_edges:
        graph.add_edge(edge['source'], edge['target'], edge_type=edge['type'])

    positions = nx.spring_layout(graph, dim=3, seed=42, iterations=100)

    edge_x, edge_y, edge_z = [], [], []
    for source, target in graph.edges():
        x0, y0, z0 = positions[source]
        x1, y1, z1 = positions[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    edge_trace = go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        line=dict(width=1, color='rgba(130,130,130,0.35)'),
        hoverinfo='none',
    )

    colors = {
        'paper': '#3B82F6',
        'author': '#10B981',
        'topic': '#F59E0B',
        'method': '#EF4444',
        'data_source': '#8B5CF6',
    }

    node_traces = []
    for node_type, color in colors.items():
        ids = [node_id_value for node_id_value in graph.nodes()
               if node_lookup[node_id_value]['type'] == node_type]
        if not ids:
            continue
        node_traces.append(go.Scatter3d(
            x=[positions[node_id_value][0] for node_id_value in ids],
            y=[positions[node_id_value][1] for node_id_value in ids],
            z=[positions[node_id_value][2] for node_id_value in ids],
            mode='markers',
            name=node_type.replace('_', ' ').title(),
            text=[node_lookup[node_id_value]['label'] for node_id_value in ids],
            hovertemplate='%{text}<extra></extra>',
            marker=dict(size=8 if node_type != 'paper' else 5, color=color, opacity=0.85),
        ))

    figure = go.Figure(data=[edge_trace, *node_traces])
    figure.update_layout(
        title=f'NBER Knowledge Graph: {focus_topic.replace("_", " " ).title()}',
        showlegend=True,
        margin=dict(l=0, r=0, t=50, b=0),
        scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False),
    )
    figure.show()

## Next analytical steps

Useful extensions include community detection, centrality analysis, coauthor networks, topic co-occurrence, temporal slices, and semantic-similarity links based on embeddings.